# Run the recent methods on Porto data

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Run the cells in order. If a cell fails, copy the error message into the team chat (do not skip ahead).

Status of this notebook: the data conversion and baseline parts were tested; the training cells (DiffMM, HSTGMatch) were written from reading the repositories and have **not** been run by the author. If they do not work, that is expected to be fixable or, in the worst case, you write the report from the literature (see the team guide).

## 0. Check the GPU

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| GPU available:', torch.cuda.is_available())

## 1. Save results to Google Drive (so a disconnect does not lose work)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/cm4604_cp3

## 2. DiffMM (recent method 1): get the code and install

Colab already has PyTorch. We install the other libraries **without version pins** first (the repo asks for Python 3.11 and torch 2.4.0, which Colab may not have).

In [ ]:
!git clone https://github.com/decisionintelligence/DiffMM.git
!pip install -q rtree geopandas networkx einops tqdm

## 3. DiffMM quick test (1 epoch)

The first run builds cached data files, which can take a while. If this finishes and prints `test_accuracy`, the setup works. `--batch_size 128` is smaller than the README's 512 to avoid running out of GPU memory on a T4.

In [ ]:
!cd DiffMM && python main.py --city porto --keep_ratio 0.1 --epochs 1 --batch_size 128 --gpu_id 0 --train_flag --test_flag

### If step 3 fails: pinned environment (Python 3.11, as in the README)

Only run this cell if step 3 fails with import or version errors.

In [ ]:
!pip install -q uv
!uv venv --python 3.11 /content/py311
!uv pip install --python /content/py311/bin/python 'numpy<2' torch==2.4.0 rtree==1.0.1 geopandas==0.14.4 networkx==3.3 einops==0.8.0 tqdm
!cd DiffMM && /content/py311/bin/python main.py --city porto --keep_ratio 0.1 --epochs 1 --batch_size 128 --gpu_id 0 --train_flag --test_flag

## 4. DiffMM real runs

Run **r = 0.1 (about 150 s gap) first**, then **r = 0.025 (about 600 s)**. Do the others only if time allows. 10 epochs is a starting point; the code has early stopping. Note the time each run takes for your report.

In [ ]:
%%time
!cd DiffMM && python main.py --city porto --keep_ratio 0.1 --epochs 10 --batch_size 128 --gpu_id 0 --train_flag --test_flag 2>&1 | tee ../diffmm_r0.1.txt

In [ ]:
%%time
!cd DiffMM && python main.py --city porto --keep_ratio 0.025 --epochs 10 --batch_size 128 --gpu_id 0 --train_flag --test_flag 2>&1 | tee ../diffmm_r0.025.txt

In [ ]:
# optional, if time allows
# !cd DiffMM && python main.py --city porto --keep_ratio 0.2 --epochs 10 --batch_size 128 --gpu_id 0 --train_flag --test_flag 2>&1 | tee ../diffmm_r0.2.txt
# !cd DiffMM && python main.py --city porto --keep_ratio 0.05 --epochs 10 --batch_size 128 --gpu_id 0 --train_flag --test_flag 2>&1 | tee ../diffmm_r0.05.txt

### Collect the DiffMM results and save them

In [ ]:
!grep -h 'test_accuracy' diffmm_r*.txt
!cp diffmm_r*.txt /content/drive/MyDrive/cm4604_cp3/
!cp -r DiffMM/model /content/drive/MyDrive/cm4604_cp3/diffmm_model

**Expected:** the DiffMM paper reports about 86% accuracy at the 150 s gap when trained on 16,000 Porto trips (the size of this toy training set). Mid-80s is a healthy result. If you see below 50%, something is wrong; tell the group before writing anything.

## 5. Baselines: HMM and snapping on the same 10,000 test trips

Upload `mm_experiment.py` (from the team guide) to the Colab file panel first.

In [ ]:
!pip install -q shapely scipy matplotlib
!python mm_experiment.py --root DiffMM/data/porto --split test --n 10000 --sigma 10 --beta 50 2>&1 | tee baseline_hmm.txt
!cp baseline_hmm.txt results.csv results.png /content/drive/MyDrive/cm4604_cp3/

## 6. HSTGMatch (recent method 2): only after DiffMM works

The paper's data are private, so we convert Porto into its input format. Upload `porto_to_hstgmatch.py` to the Colab file panel first.

In [ ]:
!git clone https://github.com/Nerooo-g/HSTGMatch.git
!pip install -q torch_geometric pyyaml scikit-learn
!python porto_to_hstgmatch.py --porto DiffMM/data/porto --out HSTGMatch/data/porto_r0.1 --keep_ratio 0.1 --n_train 6000 --n_extra 2000

In [ ]:
# smoke test that the repo itself runs (uses its own synthetic data)
!cd HSTGMatch && python scripts/smoke_test.py

In [ ]:
%%time
# stage 1: pre-training (short)
!cd HSTGMatch && python scripts/pretrain.py --config configs/default.yaml --data_dir data/porto_r0.1 --epochs 5 --batch_size 32 2>&1 | tee ../hstg_pretrain.txt

In [ ]:
%%time
# stage 2: supervised training (short)
!cd HSTGMatch && python scripts/train.py --config configs/default.yaml --data_dir data/porto_r0.1 --pretrained_encoder checkpoints/pretrain/best_encoder.pt --epochs 10 --batch_size 32 2>&1 | tee ../hstg_train.txt

In [ ]:
!cp hstg_*.txt /content/drive/MyDrive/cm4604_cp3/

## 7. What to write down (copy into the team notes)

- Date, who ran it, runtime type (T4), Python and torch versions
- The exact command for each run, the number of epochs, the time each run took
- Every `test_accuracy` / precision / recall / F1 line, **plus any error message and what you did about it**
- Anything you changed in the code (even one line)